In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Business_Feature_Aggregation") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020") \
    .getOrCreate()

In [2]:
spark

In [3]:
df = spark.read.parquet(
    "hdfs://namenode:8020/google_local/sentiment_analysis/state=California"
)

In [4]:
df.printSchema()
df.show(10, truncate=False)

root
 |-- gmap_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- category: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- avg_rating: double (nullable = true)
 |-- num_of_reviews: long (nullable = true)
 |-- rating: long (nullable = true)
 |-- review_date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- sentiment_label: string (nullable = true)
 |-- state: string (nullable = true)

+-------------------------------------+-------------------+--------+----------+--------------+------+-----------+----+---------------+------------------+
|gmap_id                              |name               |category|avg_rating|num_of_reviews|rating|review_date|year|sentiment_label|state             |
+-------------------------------------+-------------------+--------+----------+--------------+------+-----------+----+---------------+------------------+
|0x80dd33757c81873d:0xc16aa0ca451c9d74|George's 50'S Diner|[Diner] |4.4       |375    

In [5]:
from pyspark.sql.functions import when

df = df.withColumn(
    "sentiment_score",
    when(df.sentiment_label == "positive", 1)
    .when(df.sentiment_label == "negative", -1)
    .otherwise(0)
)

In [6]:
df.groupBy("sentiment_label", "sentiment_score") \
  .count() \
  .show()

+---------------+---------------+--------+
|sentiment_label|sentiment_score|   count|
+---------------+---------------+--------+
|       positive|              1|23571835|
|             na|              0|  616393|
|       negative|             -1|14733035|
+---------------+---------------+--------+



In [7]:
from pyspark.sql.functions import (
    count, avg, sum as _sum
)

business_features = df.groupBy(
    "gmap_id",
    "name",
    "category",
    "state"
).agg(
    avg("rating").alias("avg_user_rating"),
    count("*").alias("total_reviews"),
    avg("sentiment_score").alias("avg_sentiment_score"),
    (
        _sum(when(df.sentiment_score == -1, 1).otherwise(0)) /
        count("*")
    ).alias("negative_review_ratio")
)

In [8]:
business_features = business_features.fillna({
    "avg_user_rating": 0,
    "avg_sentiment_score": 0,
    "negative_review_ratio": 0,
    "total_reviews": 0
})

In [9]:
from pyspark.sql.functions import when

business_features = business_features.withColumn(
    "label",
    when(business_features.state == "Permanently closed", 1)
    .otherwise(0)
)

In [10]:
business_features.printSchema()

root
 |-- gmap_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- category: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- state: string (nullable = true)
 |-- avg_user_rating: double (nullable = false)
 |-- total_reviews: long (nullable = false)
 |-- avg_sentiment_score: double (nullable = false)
 |-- negative_review_ratio: double (nullable = false)
 |-- label: integer (nullable = false)



In [11]:
business_features.show(2, truncate=False)

+-------------------------------------+--------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+------------------+-------------+-------------------+---------------------+-----+
|gmap_id                              |name                      |category                                                                                                                                                            |state             |avg_user_rating   |total_reviews|avg_sentiment_score|negative_review_ratio|label|
+-------------------------------------+--------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+------------------+-------------+-------------------+---------------------+-----+
|0x8

In [12]:
business_features.write \
    .mode("overwrite") \
    .parquet("hdfs://namenode:8020/google_local/California_ml_features")

In [13]:
business_features.select("gmap_id").distinct().count()

504404

In [14]:
business_features.groupBy("label").count().show()

+-----+------+
|label| count|
+-----+------+
|    1| 32962|
|    0|471442|
+-----+------+

